In [1]:
import os
import glob
import re

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid")

# ============================================================
# LOAD RESULTS
# ============================================================

RESULTS_PATH = "./results/CPU_data"

files = glob.glob(os.path.join(RESULTS_PATH, "scale_free*.csv"))

print("Loaded files:")
for file in files:
    print(os.path.basename(file))

dfs = []

for file in files:
    df_tmp = pd.read_csv(file)

    filename = os.path.basename(file)

    # Source filename
    df_tmp["source_file"] = filename

    # Extract CPU value from e.g.
    # scale_free_n500_array_20516353_cpu-2.0.csv
    cpu_match = re.search(r"cpu-(\d+(?:\.\d+)?)", filename)

    if cpu_match is None:
        raise ValueError(f"Could not extract CPU value from filename: {filename}")

    df_tmp["cpu"] = float(cpu_match.group(1))

    dfs.append(df_tmp)

if not dfs:
    raise FileNotFoundError(
        f"No scale_free CSV files found in: {RESULTS_PATH}"
    )

df_scale_free = pd.concat(dfs, ignore_index=True)

print(f"\nTotal rows: {len(df_scale_free)}")

display(df_scale_free.head())

print("\nColumns:")
print(df_scale_free.columns.tolist())

print("\nCPU values:")
print(sorted(df_scale_free["cpu"].unique()))

Loaded files:
scale_free_n500_array_20516353_cpu-2.0.csv
scale_free_n500_array_20516320_cpu-1.0.csv
scale_free_n800_array_20516382_cpu-0.5.csv
scale_free_n500_array_20516289_cpu-0.5.csv
scale_free_n800_array_20516376_cpu-1.0.csv
scale_free_n500_array_20507211_cpu-0.25.csv

Total rows: 50


,array_task_id,topology,number_of_islands,number_of_migrants,migration_interval,m0,m,seed,average,best,source_file,cpu
0,1,scale_free,500,5,5,35,31,123,82.566937,19.413288,scale_free_n500_array_20516353_cpu-2.0.csv,2.0
1,3,scale_free,500,5,5,35,31,123,79.434608,19.738896,scale_free_n500_array_20516353_cpu-2.0.csv,2.0
2,0,scale_free,500,5,5,35,31,123,79.560851,18.466895,scale_free_n500_array_20516353_cpu-2.0.csv,2.0
3,2,scale_free,500,5,5,35,31,123,79.800882,21.238190,scale_free_n500_array_20516353_cpu-2.0.csv,2.0
4,4,scale_free,500,5,5,35,31,123,79.179422,19.808438,scale_free_n500_array_20516353_cpu-2.0.csv,2.0



Columns:
['array_task_id', 'topology', 'number_of_islands', 'number_of_migrants', 'migration_interval', 'm0', 'm', 'seed', 'average', 'best', 'source_file', 'cpu']

CPU values:
[0.25, 0.5, 1.0, 2.0]


In [2]:
df_500 = df_scale_free[df_scale_free["number_of_islands"]==500]

df_500

,array_task_id,topology,number_of_islands,number_of_migrants,migration_interval,m0,m,seed,average,best,source_file,cpu
0,1,scale_free,500,5,5,35,31,123,82.566937,19.413288,scale_free_n500_array_20516353_cpu-2.0.csv,2.00
1,3,scale_free,500,5,5,35,31,123,79.434608,19.738896,scale_free_n500_array_20516353_cpu-2.0.csv,2.00
2,0,scale_free,500,5,5,35,31,123,79.560851,18.466895,scale_free_n500_array_20516353_cpu-2.0.csv,2.00
3,2,scale_free,500,5,5,35,31,123,79.800882,21.238190,scale_free_n500_array_20516353_cpu-2.0.csv,2.00
4,4,scale_free,500,5,5,35,31,123,79.179422,19.808438,scale_free_n500_array_20516353_cpu-2.0.csv,2.00
5,7,scale_free,500,5,5,35,31,123,69.943442,16.417131,scale_free_n500_array_20516353_cpu-2.0.csv,2.00
6,5,scale_free,500,5,5,35,31,123,70.200601,19.895819,scale_free_n500_array_20516353_cpu-2.0.csv,2.00
7,6,scale_free,500,5,5,35,31,123,70.201247,19.948503,scale_free_n500_array_20516353_cpu-2.0.csv,2.00
8,8,scale_free,500,5,5,35,31,123,75.854609,21.234302,scale_free_n500_array_20516353_cpu-2.0.csv,2.00
9,9,scale_free,500,5,5,35,31,123,66.185908,15.749306,scale_free_n500_array_20516353_cpu-2.0.csv,2.00


In [3]:
df_cpu_compare = (
    df_500.groupby(["number_of_islands", "cpu"], as_index=False).agg(
        mean_best=("best", "mean"),
        std_best=("best", "std"),
        #min_best=("best", "min"),
        median_best=("best", "median"),
        mean_avg=("average", "mean"),
        std_avg=("average", "std"),
    )
)

In [4]:
df_cpu_compare

,number_of_islands,cpu,mean_best,std_best,median_best,mean_avg,std_avg
0,500,0.25,29.125176,4.031127,28.762828,93.575513,9.476972
1,500,0.50,19.757207,1.434489,20.236227,69.160875,3.634932
2,500,1.00,20.324216,2.150832,20.422066,79.919791,9.507325
3,500,2.00,19.191077,1.832770,19.773667,75.292851,5.651290


In [5]:
from scipy.stats import kruskal

stat, p = kruskal(
    df_500[df_500["cpu"]==0.25]['best'],
    df_500[df_500["cpu"]==0.5]['best'],
    df_500[df_500["cpu"]==1.0]['best'],
    df_500[df_500["cpu"]==2.0]['best'],
)

print(stat, p)

21.462075436291997 8.439670005983937e-05


In [6]:
import scikit_posthocs as sp

posthoc = sp.posthoc_dunn(
    df_500,
    val_col="best",
    group_col="cpu",
    p_adjust="holm"
)

print(posthoc)

          0.25      0.50      1.00      2.00
0.25  1.000000  0.001614  0.004286  0.000142
0.50  0.001614  1.000000  1.000000  1.000000
1.00  0.004286  1.000000  1.000000  1.000000
2.00  0.000142  1.000000  1.000000  1.000000


In [7]:
from cliffs_delta import cliffs_delta


delta, interpretation = cliffs_delta(
    df_500[df_500["cpu"]==0.25]['best'],
    df_500[df_500["cpu"]==1.0]['best'],
)

print(delta)
print(interpretation)

0.94
large


In [8]:
# ============================================================
# LOAD RESULTS
# ============================================================

RESULTS_PATH = "results/Sphere"

files = glob.glob(os.path.join(RESULTS_PATH, "*.csv"))

print("Loaded files:")
for f in files:
    print(os.path.basename(f))

dfs = []

for file in files:
    df_tmp = pd.read_csv(file)

    # source filename
    df_tmp["source_file"] = os.path.basename(file)

    dfs.append(df_tmp)

df_sphere = pd.concat(dfs, ignore_index=True)

print(f"\nTotal rows: {len(df_sphere)}")

display(df_sphere.head())

print("\nColumns:")
print(df_sphere.columns.tolist())

Loaded files:
complete_n500_array_20754047.csv
meeting_n500_array_20753768.csv
scale_free_n500_array_20753349.csv

Total rows: 30


,array_task_id,topology,number_of_islands,number_of_migrants,migration_interval,average,best,source_file,z_star,n_steps,r0,r1,gamma,build_target_ratio,seed,m0,m
0,0,complete,500,5,5,63.283869,43.779620,complete_n500_array_20754047.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,complete,500,5,5,49.610515,36.915965,complete_n500_array_20754047.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,complete,500,5,5,49.022557,36.915965,complete_n500_array_20754047.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2,complete,500,5,5,48.969388,36.915965,complete_n500_array_20754047.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,complete,500,5,5,54.130140,37.712196,complete_n500_array_20754047.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Columns:
['array_task_id', 'topology', 'number_of_islands', 'number_of_migrants', 'migration_interval', 'average', 'best', 'source_file', 'z_star', 'n_steps', 'r0', 'r1', 'gamma', 'build_target_ratio', 'seed', 'm0', 'm']


In [9]:
df_sphere_compare = (
    df_sphere.groupby(["number_of_islands", "topology"], as_index=False).agg(
        mean_best=("best", "mean"),
        std_best=("best", "std"),
        median_best=("best", "median"),
        mean_avg=("average", "mean"),
        std_avg=("average", "std"),
    )
)

In [10]:
df_sphere_compare

,number_of_islands,topology,mean_best,std_best,median_best,mean_avg,std_avg
0,500,complete,38.994767,3.548541,37.314081,52.022784,4.360081
1,500,meeting,47.304480,3.133655,47.704929,62.651471,3.344370
2,500,scale_free,19.409474,2.486326,18.292773,79.404202,4.892397


In [11]:
stat, p = kruskal(
    df_sphere[df_sphere['topology']=='complete']['best'],
    df_sphere[df_sphere['topology']=='meeting']['best'],
    df_sphere[df_sphere['topology']=='scale_free']['best']
)

print(stat, p)

posthoc = sp.posthoc_dunn(
    df_sphere,
    val_col="best",
    group_col="topology",
    p_adjust="holm"
)

print(posthoc)

24.92082216264523 3.877146622975739e-06
            complete   meeting  scale_free
complete    1.000000  0.019193    0.016232
meeting     0.019193  1.000000    0.000002
scale_free  0.016232  0.000002    1.000000


In [12]:
delta, interpretation = cliffs_delta(
    df_sphere[df_sphere['topology']=='complete']['best'],
    df_sphere[df_sphere['topology']=='meeting']['best'],
)

print(delta)
print(interpretation)

-0.92
large


In [13]:
delta, interpretation = cliffs_delta(
    df_sphere[df_sphere['topology']=='meeting']['best'],
    df_sphere[df_sphere['topology']=='scale_free']['best']
)

print(delta)
print(interpretation)



1.0
large


In [14]:
delta, interpretation = cliffs_delta(
    df_sphere[df_sphere['topology']=='complete']['best'],
    df_sphere[df_sphere['topology']=='scale_free']['best']
)

print(delta)
print(interpretation)

1.0
large


In [15]:

# ============================================================
# LOAD RESULTS
# ============================================================`

for path in ["results/Sphere", "results/Labs", "results/800_islands_data"]:

    print(f"\n === {path} === ")

    RESULTS_PATH = path

    files = glob.glob(os.path.join(RESULTS_PATH, "*.csv"))

    print("Loaded files:")
    for f in files:
        print(os.path.basename(f))

    dfs = []

    for file in files:
        df_tmp = pd.read_csv(file)

        # source filename
        df_tmp["source_file"] = os.path.basename(file)

        dfs.append(df_tmp)

    df = pd.concat(dfs, ignore_index=True)

    print(f"\nTotal rows: {len(df_sphere)}")

    display(df.head())

    print("\nColumns:")
    print(df.columns.tolist())
    df_compare = (
        df.groupby(["number_of_islands", "topology"], as_index=False).agg(
            mean_best=("best", "mean"),
            std_best=("best", "std"),
            median_best=("best", "median"),
            mean_avg=("average", "mean"),
            std_avg=("average", "std"),
        )
    )

    display(df_compare)

    stat, p = kruskal(
        df[df['topology'] == 'complete']['best'],
        df[df['topology'] == 'meeting']['best'],
        df[df['topology'] == 'scale_free']['best']
    )

    print(stat, p)

    posthoc = sp.posthoc_dunn(
        df,
        val_col="best",
        group_col="topology",
        p_adjust="holm"
    )

    print(posthoc)
    delta, interpretation = cliffs_delta(
        df[df['topology'] == 'complete']['best'],
        df[df['topology'] == 'meeting']['best'],
    )

    print(delta)
    print(interpretation)
    delta, interpretation = cliffs_delta(
        df[df['topology'] == 'meeting']['best'],
        df[df['topology'] == 'scale_free']['best']
    )

    print(delta)
    print(interpretation)

    delta, interpretation = cliffs_delta(
        df[df['topology'] == 'complete']['best'],
        df[df['topology'] == 'scale_free']['best']
    )

    print(delta)
    print(interpretation)


 === results/Sphere === 
Loaded files:
complete_n500_array_20754047.csv
meeting_n500_array_20753768.csv
scale_free_n500_array_20753349.csv

Total rows: 30


,array_task_id,topology,number_of_islands,number_of_migrants,migration_interval,average,best,source_file,z_star,n_steps,r0,r1,gamma,build_target_ratio,seed,m0,m
0,0,complete,500,5,5,63.283869,43.779620,complete_n500_array_20754047.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,complete,500,5,5,49.610515,36.915965,complete_n500_array_20754047.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,complete,500,5,5,49.022557,36.915965,complete_n500_array_20754047.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2,complete,500,5,5,48.969388,36.915965,complete_n500_array_20754047.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,complete,500,5,5,54.130140,37.712196,complete_n500_array_20754047.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Columns:
['array_task_id', 'topology', 'number_of_islands', 'number_of_migrants', 'migration_interval', 'average', 'best', 'source_file', 'z_star', 'n_steps', 'r0', 'r1', 'gamma', 'build_target_ratio', 'seed', 'm0', 'm']


,number_of_islands,topology,mean_best,std_best,median_best,mean_avg,std_avg
0,500,complete,38.994767,3.548541,37.314081,52.022784,4.360081
1,500,meeting,47.304480,3.133655,47.704929,62.651471,3.344370
2,500,scale_free,19.409474,2.486326,18.292773,79.404202,4.892397


24.92082216264523 3.877146622975739e-06
            complete   meeting  scale_free
complete    1.000000  0.019193    0.016232
meeting     0.019193  1.000000    0.000002
scale_free  0.016232  0.000002    1.000000
-0.92
large
1.0
large
1.0
large

 === results/Labs === 
Loaded files:
meeting_n500_array_20754340.csv
scale_free_n500_array_20754328.csv
complete_n500_array_20754310.csv

Total rows: 30


...

23.29554764024935 8.738484162490832e-06
            complete   meeting  scale_free
complete    1.000000  0.047495    0.009594
meeting     0.047495  1.000000    0.000005
scale_free  0.009594  0.000005    1.000000
-0.78
large
1.0
large
1.0
large

 === results/800_islands_data === 
Loaded files:
meeting_n800_array_20528691.csv
scale_free_n800_array_20516376_cpu-1.0.csv
complete_n800_array_20527009.csv

Total rows: 30


,array_task_id,topology,number_of_islands,number_of_migrants,migration_interval,z_star,n_steps,r0,r1,gamma,build_target_ratio,seed,average,best,source_file,m0,m
0,0,meeting,800,5,5,20.0,2000.0,0.001,1.0,0.02,0.8,123.0,37.606532,29.384320,meeting_n800_array_20528691.csv,NaN,NaN
1,2,meeting,800,5,5,20.0,2000.0,0.001,1.0,0.02,0.8,123.0,37.479787,27.302521,meeting_n800_array_20528691.csv,NaN,NaN
2,3,meeting,800,5,5,20.0,2000.0,0.001,1.0,0.02,0.8,123.0,41.413155,29.891581,meeting_n800_array_20528691.csv,NaN,NaN
3,1,meeting,800,5,5,20.0,2000.0,0.001,1.0,0.02,0.8,123.0,31.239294,23.582410,meeting_n800_array_20528691.csv,NaN,NaN
4,4,meeting,800,5,5,20.0,2000.0,0.001,1.0,0.02,0.8,123.0,45.786709,33.984015,meeting_n800_array_20528691.csv,NaN,NaN



Columns:
['array_task_id', 'topology', 'number_of_islands', 'number_of_migrants', 'migration_interval', 'z_star', 'n_steps', 'r0', 'r1', 'gamma', 'build_target_ratio', 'seed', 'average', 'best', 'source_file', 'm0', 'm']


,number_of_islands,topology,mean_best,std_best,median_best,mean_avg,std_avg
0,800,complete,28.449560,3.385578,25.988512,35.456789,5.735959
1,800,meeting,28.828969,3.803567,29.384320,38.705095,5.383522
2,800,scale_free,9.510355,1.412792,8.878535,57.290909,6.084387


9.661538461538472 0.00798038015032894
            complete   meeting  scale_free
complete    1.000000  0.829896    0.019874
meeting     0.829896  1.000000    0.015674
scale_free  0.019874  0.015674    1.000000
-0.12
negligible
1.0
large
1.0
large


In [16]:
df_compare

,number_of_islands,topology,mean_best,std_best,median_best,mean_avg,std_avg
0,800,complete,28.449560,3.385578,25.988512,35.456789,5.735959
1,800,meeting,28.828969,3.803567,29.384320,38.705095,5.383522
2,800,scale_free,9.510355,1.412792,8.878535,57.290909,6.084387


In [17]:
df_sphere_compare

,number_of_islands,topology,mean_best,std_best,median_best,mean_avg,std_avg
0,500,complete,38.994767,3.548541,37.314081,52.022784,4.360081
1,500,meeting,47.304480,3.133655,47.704929,62.651471,3.344370
2,500,scale_free,19.409474,2.486326,18.292773,79.404202,4.892397


In [18]:
df

,array_task_id,topology,number_of_islands,number_of_migrants,migration_interval,z_star,n_steps,r0,r1,gamma,build_target_ratio,seed,average,best,source_file,m0,m
0,0,meeting,800,5,5,20.0,2000.0,0.001,1.0,0.02,0.8,123.0,37.606532,29.384320,meeting_n800_array_20528691.csv,NaN,NaN
1,2,meeting,800,5,5,20.0,2000.0,0.001,1.0,0.02,0.8,123.0,37.479787,27.302521,meeting_n800_array_20528691.csv,NaN,NaN
2,3,meeting,800,5,5,20.0,2000.0,0.001,1.0,0.02,0.8,123.0,41.413155,29.891581,meeting_n800_array_20528691.csv,NaN,NaN
3,1,meeting,800,5,5,20.0,2000.0,0.001,1.0,0.02,0.8,123.0,31.239294,23.582410,meeting_n800_array_20528691.csv,NaN,NaN
4,4,meeting,800,5,5,20.0,2000.0,0.001,1.0,0.02,0.8,123.0,45.786709,33.984015,meeting_n800_array_20528691.csv,NaN,NaN
5,2,scale_free,800,5,5,NaN,NaN,NaN,NaN,NaN,NaN,123.0,54.572589,8.878535,scale_free_n800_array_20516376_cpu-1.0.csv,35.0,31.0
6,3,scale_free,800,5,5,NaN,NaN,NaN,NaN,NaN,NaN,123.0,54.569320,8.878535,scale_free_n800_array_20516376_cpu-1.0.csv,35.0,31.0
7,1,scale_free,800,5,5,NaN,NaN,NaN,NaN,NaN,NaN,123.0,54.568962,8.878535,scale_free_n800_array_20516376_cpu-1.0.csv,35.0,31.0
8,0,scale_free,800,5,5,NaN,NaN,NaN,NaN,NaN,NaN,123.0,54.568683,8.878535,scale_free_n800_array_20516376_cpu-1.0.csv,35.0,31.0
9,4,scale_free,800,5,5,NaN,NaN,NaN,NaN,NaN,NaN,123.0,68.174992,12.037633,scale_free_n800_array_20516376_cpu-1.0.csv,35.0,31.0


# Analiza statystyczna 500 vs 800

In [19]:
import os
import glob
import pandas as pd

RESULTS_PATH = "results/best_configs_500"

files_500 = {
    "meeting": "meeting_n500_array_20500107.csv",
    "scale_free": "scale_free_n500_array_20500054.csv",
    "complete": "complete_n500_array_20500075.csv",
}

dfs = []

for topology, filename in files_500.items():
    path = os.path.join(RESULTS_PATH, filename)

    df_tmp = pd.read_csv(path)

    df_tmp["topology"] = topology
    df_tmp["islands"] = 500
    df_tmp["source_file"] = filename

    dfs.append(df_tmp)

df_500 = pd.concat(dfs, ignore_index=True)

print(df_500.groupby("topology").size())
display(df_500.head())

topology
complete      20
meeting       20
scale_free    20
dtype: int64


,array_task_id,topology,number_of_islands,number_of_migrants,migration_interval,z_star,n_steps,r0,r1,gamma,build_target_ratio,seed,average,best,islands,source_file,m0,m
0,3,meeting,500,5,5,20.0,2000.0,0.001,1.0,0.02,0.8,123.0,54.109538,37.294479,500,meeting_n500_array_20500107.csv,NaN,NaN
1,0,meeting,500,5,5,20.0,2000.0,0.001,1.0,0.02,0.8,123.0,61.239845,40.099496,500,meeting_n500_array_20500107.csv,NaN,NaN
2,1,meeting,500,5,5,20.0,2000.0,0.001,1.0,0.02,0.8,123.0,55.899318,37.281817,500,meeting_n500_array_20500107.csv,NaN,NaN
3,2,meeting,500,5,5,20.0,2000.0,0.001,1.0,0.02,0.8,123.0,56.665015,38.876094,500,meeting_n500_array_20500107.csv,NaN,NaN
4,4,meeting,500,5,5,20.0,2000.0,0.001,1.0,0.02,0.8,123.0,51.675822,37.492602,500,meeting_n500_array_20500107.csv,NaN,NaN


In [20]:
df_800 = df.copy()
df_800["islands"] = 800
print(df_800.groupby("topology").size())

topology
complete      5
meeting       5
scale_free    5
dtype: int64


In [21]:
df_comparison = pd.concat(
    [df_500, df_800],
    ignore_index=True
)

print(
    df_comparison.groupby(["topology", "islands"]).size()
)

topology    islands
complete    500        20
            800         5
meeting     500        20
            800         5
scale_free  500        20
            800         5
dtype: int64


In [22]:
df_comparison["group"] = (
    df_comparison["topology"]
    + "_"
    + df_comparison["islands"].astype(str)
)

In [25]:
def cliffs_delta(x, y):
    x = list(x)
    y = list(y)

    greater = 0
    lower = 0

    for xi in x:
        for yi in y:
            if xi > yi:
                greater += 1
            elif xi < yi:
                lower += 1

    return (greater - lower) / (len(x) * len(y))

In [26]:
def delta_magnitude(delta):
    d = abs(delta)

    if d < 0.147:
        return "negligible"
    elif d < 0.330:
        return "small"
    elif d < 0.474:
        return "medium"
    else:
        return "large"

for topology in ["complete", "meeting", "scale_free"]:

    x500 = df_comparison[
        (df_comparison["topology"] == topology) &
        (df_comparison["islands"] == 500)
    ]["best"]

    x800 = df_comparison[
        (df_comparison["topology"] == topology) &
        (df_comparison["islands"] == 800)
    ]["best"]

    H, p = kruskal(x500, x800)
    delta = cliffs_delta(x500, x800)

    print(
        f"{topology:12s}: "
        f"H={H:.3f}, "
        f"p={p:.6f}, "
        f"delta={delta:.3f}, "
        f"effect={delta_magnitude(delta)}"
    )

complete    : H=11.570, p=0.000670, delta=1.000, effect=large
meeting     : H=11.538, p=0.000682, delta=1.000, effect=large
scale_free  : H=11.587, p=0.000664, delta=1.000, effect=large


In [27]:
for topology in ["complete", "meeting", "scale_free"]:
    x500 = df_comparison[
        (df_comparison["topology"] == topology) &
        (df_comparison["islands"] == 500)
    ]["best"]

    x800 = df_comparison[
        (df_comparison["topology"] == topology) &
        (df_comparison["islands"] == 800)
    ]["best"]

    print(topology)
    print("500:", x500.min(), "-", x500.max())
    print("800:", x800.min(), "-", x800.max())

complete
500: 35.93233301361491 - 51.35120539285691
800: 25.988512376683214 - 32.6009614225523
meeting
500: 37.2818170097483 - 53.49624608263856
800: 23.58241004415421 - 33.98401499029121
scale_free
500: 16.90492187490409 - 24.161859646234312
800: 8.878534858839389 - 12.037633217108704
